# Modeling dynamics of FS Peptide
This example shows a typical, basic usage of the MSMBuilder command line to model dynamics of a protein system. 

In [ ]:
# Work in a temporary directory
import tempfile
import os
os.chdir(tempfile.mkdtemp())

In [ ]:
# Since this is running from an IPython notebook,
# we prefix all our commands with "!"
# When running on the command line, omit the leading "!"
! msmb -h

### Get example data

In [ ]:
! msmb FsPeptide
# The data is downloaded once, into MSMBuilder's data directory; link it here.
import os
from msmbuilder.example_datasets import get_data_home
os.symlink(os.path.join(get_data_home(), 'fs_peptide'), 'fs_peptide')
! ls fs_peptide | head

### Featurization
The raw `(x, y, z)` coordinates from the simulation do not respect the translational and rotational symmetry of our problem. A `Featurizer` transforms cartesian coordinates into other representations. Here we use the `DihedralFeaturizer` to turn our data into phi and psi dihedral angles. Observe that the `264*3`-dimensional space is reduced to 84 dimensions.

In [ ]:
# Remember '\' is the line-continuation marker
# You can enter this command on one line
! msmb DihedralFeaturizer \
    --out featurizer.pkl  \
    --transformed diheds  \
    --top fs_peptide/fs-peptide.pdb \
    --trjs "fs_peptide/*.xtc" \
    --stride 10

### Preprocessing
Since the range of values in our raw data can vary widely from feature to feature, we can scale values to reduce bias. Here we use the `RobustScaler` to center and scale our dihedral angles by their respective interquartile ranges.

In [ ]:
! msmb RobustScaler \
    -i diheds \
    --transformed scaled_diheds.h5

### Intermediate kinetic model: tICA
`tICA` is similar to principal component analysis (see "tICA vs. PCA" example). Note that the 84-dimensional space is reduced to 4 dimensions.

In [ ]:
! msmb tICA -i scaled_diheds.h5 \
    --out tica_model.pkl \
    --transformed tica_trajs.h5 \
    --n_components 4 \
    --lag_time 2

### tICA Histogram
We can histogram our data projecting along the two slowest degrees of freedom (as found by tICA). You have to do this in a python script.

In [ ]:
from msmbuilder.dataset import dataset
ds = dataset('tica_trajs.h5')

%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt

txx = np.concatenate(list(ds))

def plot_free_energy(x, y, weights=None, bins=80, ax=None):
    """Free energy -log(p), in kT, of a 2-D histogram; minimum shifted to zero."""
    ax = ax or plt.gca()
    H, xe, ye = np.histogram2d(x, y, bins=bins, weights=weights)
    with np.errstate(divide='ignore'):
        F = -np.log(H.T / H.sum())
    F -= F[np.isfinite(F)].min()
    xc, yc = (xe[1:] + xe[:-1]) / 2, (ye[1:] + ye[:-1]) / 2
    filled = ax.contourf(xc, yc, F, levels=np.linspace(0, 8, 17), cmap='viridis', extend='max')
    plt.colorbar(filled, ax=ax, label='Free energy ($k_BT$)')
    return ax

fig, ax = plt.subplots()
for i in range(txx.shape[1]):
    ax.hist(txx[:, i], bins=100, histtype='step', label=f'tIC {i + 1}')
ax.set_xlabel('tIC value')
ax.set_ylabel('Frames')
ax.legend()

### Clustering
Conformations need to be clustered into states (sometimes written as microstates). We cluster based on the tICA projections to group conformations that interconvert rapidly. Note that we transform our trajectories from the 4-dimensional tICA space into a 1-dimensional cluster index.

In [ ]:
! msmb MiniBatchKMeans -i tica_trajs.h5 \
    --transformed labeled_trajs.h5 \
    --out clusterer.pkl \
    --n_clusters 100 \
    --random_state 42

### MSM
We can construct an MSM from the labeled trajectories

In [ ]:
! msmb MarkovStateModel -i labeled_trajs.h5 \
    --out msm.pkl \
    --lag_time 2

### Plot Free Energy Landscape
Subsequent plotting and analysis should be done from Python

In [ ]:
from msmbuilder.utils import load
msm = load('msm.pkl')
clusterer = load('clusterer.pkl')

# MSM-reweighted free energy: the frames of each MSM state together carry
# that state's equilibrium population.
labels = clusterer.partial_transform(txx)
states = np.asarray(msm.partial_transform(labels, mode='fill'), dtype=float)
ok = np.isfinite(states)
states = states[ok].astype(int)
weights = msm.populations_[states] / np.bincount(states, minlength=msm.n_states_)[states]

plot_free_energy(txx[ok, 0], txx[ok, 1], weights=weights)
plt.scatter(clusterer.cluster_centers_[msm.state_labels_, 0],
            clusterer.cluster_centers_[msm.state_labels_, 1],
            s=1e4 * msm.populations_,       # size by population
            c=msm.left_eigenvectors_[:, 1], # color by eigenvector
            cmap='coolwarm', edgecolors='k', zorder=3)
plt.colorbar(label='First dynamical eigenvector')
plt.xlabel('tIC 1')
plt.ylabel('tIC 2')
plt.tight_layout()